# Chapter 3.1 Flow Matching From Scratch

**TL;DR** — Builds the toy conditional flow matching example from first principles. The reader finishes with visible CFM batch construction, regression loss, training loop, Euler sampler, and the Chapter 3 toy figures saved under `figures/ch03`.

**Prerequisites** - Chapter 2 (distribution transport background).

**Outputs**
- `figures/ch03/fig_toy_endpoint_distributions.png`
- `figures/ch03/fig_toy_endpoint_distributions.pdf`
- `figures/ch03/fig_toy_loss.png`
- `figures/ch03/fig_toy_loss.pdf`
- `figures/ch03/fig_toy_evolution.png`
- `figures/ch03/fig_toy_evolution.pdf`
- `figures/ch03/fig03_02_conditional_vs_marginal_toy.png`
- `figures/ch03/fig03_02_conditional_vs_marginal_toy.pdf`
- `figures/ch03/fig03_03_cfm_object_hierarchy_toy.png`
- `figures/ch03/fig03_03_cfm_object_hierarchy_toy.pdf`

**Runtime** - `QUICK_MODE` defaults on and reduces toy sample count and training steps; `SMOKE_MODE` further cuts the toy dataset, training steps, batch size, and Euler steps for CI. On CPU, the quick run is a few minutes, while smoke mode is intended to finish much faster.

**Key parameters**
- `SEED` defaults to `42` via `CH03_SEED` and controls toy samples, pair batches, and subsampling.
- `QUICK_MODE` defaults to `CH03_QUICK=1`; `SMOKE_MODE` uses `CH03_SMOKE_MODE=1`.
- `toy_n = 1500` in quick mode or `5000` in full mode, with a smaller smoke cap.
- `toy_steps = 550` in quick mode or `2200` in full mode; smoke uses `35`.
- `toy_batch_size = 256`, or `96` in smoke mode.
- `toy_euler_steps = 100`, or `25` in smoke mode.

## Tutorial setup

This short setup cell resolves the project root, imports reusable plotting/reporting helpers, and chooses a reproducible run mode. It intentionally keeps path handling out of the tutorial narrative after this point.

In [ ]:
from pathlib import Path
import os
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

try:
    import torch
except ImportError as exc:
    raise ImportError("This notebook requires PyTorch for CFM training and sampling.") from exc

import sys

ROOT_HINT = Path.cwd().resolve()
if not (ROOT_HINT / "src").is_dir():
    ROOT_HINT = ROOT_HINT.parent
if str(ROOT_HINT) not in sys.path:
    sys.path.insert(0, str(ROOT_HINT))

from src.tutorial_init import apply_tutorial_plot_style, bootstrap, make_ch03_run_config, make_save_and_show

SEED = int(os.environ.get("CH03_SEED", "42"))
QUICK_MODE = os.environ.get("CH03_QUICK", "1") == "1"
boot = bootstrap(chapter="ch03", seed=SEED, quick_mode=QUICK_MODE)
PROJECT_ROOT = boot.project_root
FIG_DIR = boot.fig_dir
OUT_DIR = boot.out_dir
DEVICE = boot.device

from src.visualization import flow_matching as ch03
from src.utils import set_seed
from src.core.models import VelocityMLP, count_parameters


In [ ]:
config = make_ch03_run_config()
SEED = config.seed
QUICK_MODE = config.quick_mode
SMOKE_MODE = config.smoke_mode
PAPER_FIGURE_MODE = config.paper_figure_mode
DEVICE = config.device

context = ch03.make_ch03_context(PROJECT_ROOT)
FIG_DIR = context.fig_dir
TABLE_DIR = context.table_dir
OUT_DIR = context.output_dir

apply_tutorial_plot_style()
save_and_show = make_save_and_show(FIG_DIR, write_pdf=PAPER_FIGURE_MODE, save_fn=ch03.save_figure)
print(f"Chapter 3.1 setup: device={DEVICE}, seed={SEED}, quick={QUICK_MODE}, smoke={SMOKE_MODE}")


## 1. 2D Toy Sanity Check

The source distribution is a multi-modal source: eight Gaussian components arranged on a ring. The target is one rotated Gaussian cloud near the center. This simple geometry makes the toy problem useful: CFM must merge several source modes and change distribution shape, which would be hard to see if both endpoints were single Gaussians.

In [ ]:
toy_n = 1500 if QUICK_MODE else 5000
if SMOKE_MODE:
    toy_n = 350

toy_steps = 550 if QUICK_MODE else 2200
if SMOKE_MODE:
    toy_steps = 35

toy_batch_size = 256 if not SMOKE_MODE else 96
toy_log_every = 25 if not SMOKE_MODE else 5

print(
    f"Toy run: n={toy_n} per endpoint distribution, "
    f"steps={toy_steps}, batch={toy_batch_size}, log_every={toy_log_every}"
)


In [ ]:
toy_X0, toy_components = ch03.make_eight_gaussians(n=toy_n, seed=SEED)
toy_X1 = ch03.make_single_gaussian(n=toy_n, seed=SEED + 1)
toy_pair_batch_fn = ch03.make_random_pair_batch_fn(toy_X0, toy_X1, seed=SEED + 2)

distribution_summary = pd.DataFrame(
    [
        {
            "distribution": "source eight-gaussian mixture",
            "n": len(toy_X0),
            "mean_x": toy_X0[:, 0].mean(),
            "mean_y": toy_X0[:, 1].mean(),
            "std_x": toy_X0[:, 0].std(),
            "std_y": toy_X0[:, 1].std(),
            "components": int(np.unique(toy_components).size),
        },
        {
            "distribution": "target rotated Gaussian",
            "n": len(toy_X1),
            "mean_x": toy_X1[:, 0].mean(),
            "mean_y": toy_X1[:, 1].mean(),
            "std_x": toy_X1[:, 0].std(),
            "std_y": toy_X1[:, 1].std(),
            "components": 1,
        },
    ]
)
ch03.display_table(distribution_summary.round(4), n=len(distribution_summary))


### Endpoint distribution preview

The first paper-facing artifact is `fig_toy_endpoint_distributions.png`. Read it as the visual contract for the rest of the notebook: the model only sees unordered source and target samples, not observed pairings or lineage labels.

In [ ]:
preview_source_idx = ch03.subsample_idx(len(toy_X0), min(1200, len(toy_X0)), seed=SEED + 4)
preview_target_idx = ch03.subsample_idx(len(toy_X1), min(1200, len(toy_X1)), seed=SEED + 5)
xlim, ylim = ch03.robust_limits(toy_X0, toy_X1, margin=0.12)

fig, axes = plt.subplots(1, 2, figsize=(6.8, 3.2), sharex=True, sharey=True)
source_scatter = axes[0].scatter(
    toy_X0[preview_source_idx, 0],
    toy_X0[preview_source_idx, 1],
    c=toy_components[preview_source_idx],
    cmap="tab10",
    s=8,
    alpha=0.72,
    linewidths=0,
)
axes[1].scatter(
    toy_X1[preview_target_idx, 0],
    toy_X1[preview_target_idx, 1],
    color=ch03.PAPER_COLORS["target_red"],
    s=8,
    alpha=0.58,
    linewidths=0,
)
ch03.format_axis(axes[0], xlim=xlim, ylim=ylim, title="Source: eight Gaussian modes")
ch03.format_axis(axes[1], xlim=xlim, ylim=ylim, title="Target: rotated Gaussian")
axes[1].set_ylabel("")
fig.colorbar(source_scatter, ax=axes[0], fraction=0.046, pad=0.04, label="source mode")
fig.suptitle("Toy endpoint distributions", y=1.02)
fig.tight_layout()
toy_endpoint_path = save_and_show(fig, "fig_toy_endpoint_distributions.png", width=760)


## 2. Conditional path target

For a sampled pair `(x0, x1)`, the linear conditional bridge is `x_t = (1 - t) x0 + t x1`. The conditional velocity is constant along this bridge: `u_t = x1 - x0`. This constant target is what makes the from-scratch CFM objective easy to inspect before later notebooks introduce more structured couplings and diagnostics.

In [ ]:
def make_cfm_batch(pair_batch_fn, batch_size, device):
    pair_batch = pair_batch_fn(batch_size)
    x0 = torch.as_tensor(pair_batch["x0"], dtype=torch.float32, device=device)
    x1 = torch.as_tensor(pair_batch["x1"], dtype=torch.float32, device=device)
    t = torch.rand(x0.shape[0], 1, dtype=torch.float32, device=device)
    x_t = (1.0 - t) * x0 + t * x1
    target_velocity = x1 - x0
    return {"x0": x0, "x1": x1, "t": t, "x_t": x_t, "target_velocity": target_velocity}


In [ ]:
preview_batch = make_cfm_batch(toy_pair_batch_fn, 6, DEVICE)
conditional_path_preview = pd.DataFrame(
    {
        "t": ch03.as_np(preview_batch["t"]).ravel(),
        "x0_1": ch03.as_np(preview_batch["x0"])[:, 0],
        "x0_2": ch03.as_np(preview_batch["x0"])[:, 1],
        "x1_1": ch03.as_np(preview_batch["x1"])[:, 0],
        "x1_2": ch03.as_np(preview_batch["x1"])[:, 1],
        "x_t_1": ch03.as_np(preview_batch["x_t"])[:, 0],
        "x_t_2": ch03.as_np(preview_batch["x_t"])[:, 1],
        "u_t_1": ch03.as_np(preview_batch["target_velocity"])[:, 0],
        "u_t_2": ch03.as_np(preview_batch["target_velocity"])[:, 1],
    }
)
ch03.display_table(conditional_path_preview.round(3), n=len(conditional_path_preview))


## 3. Inline CFM loss

The neural field receives `(x_t, t)` and predicts a marginal velocity. The regression target in each mini-batch is the endpoint-conditioned velocity from the sampled pair. The key simplification here is that the conditional velocity is constant under linear interpolation; later transport choices change which endpoint pairs define these local targets.

In [ ]:
def cfm_loss_from_batch(model, batch):
    pred_velocity = model(batch["x_t"], batch["t"])
    loss = torch.mean((pred_velocity - batch["target_velocity"]) ** 2)
    return loss, pred_velocity


In [ ]:
toy_model = VelocityMLP(x_dim=2, hidden_dim=128, hidden_layers=4).to(DEVICE)
toy_optimizer = torch.optim.Adam(toy_model.parameters(), lr=1e-3)
initial_batch = make_cfm_batch(toy_pair_batch_fn, toy_batch_size, DEVICE)
initial_loss, _ = cfm_loss_from_batch(toy_model, initial_batch)

print(
    "VelocityMLP maps (x_t, t) to a 2D velocity; "
    f"parameters={count_parameters(toy_model):,}, initial_batch_mse={float(initial_loss.detach().cpu()):.4f}"
)


## 4. Inline toy training loop

The loop is intentionally short: make a CFM batch, compute local MSE, take one optimizer step, and log the loss. No ODE solver appears in training; this separation is the main contrast with solver-in-the-loop neural ODE baselines.

In [ ]:
def train_toy_cfm(model, pair_batch_fn, optimizer, *, n_steps, batch_size, device, log_every):
    rows = []
    start_time = time.perf_counter()
    model.train()
    for step in range(1, int(n_steps) + 1):
        batch = make_cfm_batch(pair_batch_fn, batch_size, device)
        loss, _ = cfm_loss_from_batch(model, batch)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        should_log = step == 1 or step % int(log_every) == 0 or step == int(n_steps)
        if should_log:
            elapsed = time.perf_counter() - start_time
            rows.append(
                {
                    "method": "cfm_local_regression",
                    "step": step,
                    "loss": float(loss.detach().cpu()),
                    "wall_time_sec": elapsed,
                    "sec_per_step": elapsed / step,
                    "nfe_train_per_step": 1,
                    "batch_size": int(batch_size),
                }
            )
    return pd.DataFrame(rows)


In [ ]:
toy_history = train_toy_cfm(
    toy_model,
    toy_pair_batch_fn,
    toy_optimizer,
    n_steps=toy_steps,
    batch_size=toy_batch_size,
    device=DEVICE,
    log_every=toy_log_every,
)


In [ ]:
loss_preview = toy_history.tail(8).copy()
ch03.display_table(loss_preview, n=len(loss_preview))


In [ ]:
fig, ax = plt.subplots(figsize=(4.8, 3.2))
ax.plot(toy_history["step"], toy_history["loss"], color="#2F6B5E", linewidth=1.6)
ax.set_xlabel("optimization step")
ax.set_ylabel("CFM local MSE")
ax.set_title("Toy CFM training loss")
ax.grid(axis="y", color="0.92", linewidth=0.7)
toy_loss_path = save_and_show(fig, "fig_toy_loss.png", width=620)


## 5. Inline Euler sampler

After training, sampling is inference: integrate the learned time-dependent vector field from `t=0` to `t=1`. Training never integrates this ODE, so changing the sampler budget affects inference cost but not the local regression objective used above.

In [ ]:
def euler_rollout(model, x0, *, n_steps=100, return_traj=True):
    was_training = bool(model.training)
    model.eval()
    n_steps = int(n_steps)
    dt = 1.0 / n_steps
    x = x0.detach().clone()
    trajectory = [x.detach().clone()] if return_traj else None

    with torch.no_grad():
        for step in range(n_steps):
            t = x.new_full((x.shape[0], 1), step / n_steps)
            velocity = model(x, t)
            x = x + dt * velocity
            if return_traj:
                trajectory.append(x.detach().clone())

    if was_training:
        model.train()
    traj = torch.stack(trajectory, dim=0) if return_traj else None
    return x, traj, int(n_steps)


In [ ]:
toy_sample_n = min(900 if QUICK_MODE else 1800, len(toy_X0))
if SMOKE_MODE:
    toy_sample_n = min(220, len(toy_X0))

toy_sample_idx = ch03.subsample_idx(len(toy_X0), toy_sample_n, seed=SEED + 3)
toy_x0_t = torch.as_tensor(toy_X0[toy_sample_idx], dtype=torch.float32, device=DEVICE)
toy_euler_steps = 100 if not SMOKE_MODE else 25
print(f"Euler rollout: sampled_source_particles={toy_sample_n}, steps={toy_euler_steps}, return_traj=True")


In [ ]:
toy_final_t, toy_traj_t, toy_nfe = euler_rollout(
    toy_model,
    toy_x0_t,
    n_steps=toy_euler_steps,
    return_traj=True,
)
toy_traj = ch03.as_np(toy_traj_t)


In [ ]:
print(f"Trajectory tensor shape: {tuple(toy_traj.shape)}; Euler NFE={int(toy_nfe)}")
print(
    "Population mean shift: "
    f"x {toy_traj[0, :, 0].mean():.3f} -> {toy_traj[-1, :, 0].mean():.3f}, "
    f"y {toy_traj[0, :, 1].mean():.3f} -> {toy_traj[-1, :, 1].mean():.3f}"
)


In [ ]:
toy_times = [0.0, 0.25, 0.5, 0.75, 1.0]
toy_step_idx = np.round(np.asarray(toy_times) * (toy_traj.shape[0] - 1)).astype(int)
xlim, ylim = ch03.robust_limits(toy_X0, toy_X1, toy_traj.reshape(-1, 2), margin=0.10)
fig, axes = plt.subplots(1, 5, figsize=(12.3, 2.65), sharex=True, sharey=True)
target_idx = ch03.subsample_idx(len(toy_X1), 450, seed=61)
for ax, tau, step in zip(axes, toy_times, toy_step_idx):
    pts = toy_traj[step]
    ax.scatter(toy_X1[target_idx, 0], toy_X1[target_idx, 1], s=5, color="0.72", alpha=0.20, linewidths=0)
    ax.scatter(pts[:, 0], pts[:, 1], s=6, color="#2F6B5E", alpha=0.52, linewidths=0)
    ch03.format_axis(ax, xlim, ylim, xlabel="toy x1", ylabel="toy x2", title=f"t={tau:.2f}")
    if ax is not axes[0]:
        ax.set_ylabel("")
fig.suptitle("Toy population evolution under Euler sampling")
toy_evolution_path = save_and_show(fig, "fig_toy_evolution.png", width=900)


## 6. Conditional Velocity Versus Marginal Velocity

This section corresponds to paper Figure 3.2. The diagnostic fixes a local `(x,t)` neighborhood, gathers many endpoint-conditioned velocity vectors that pass through that neighborhood, and compares their average direction with the network's single marginal velocity prediction. The plotting helper is reused from `src`, but the summary below exposes the conceptual probe.

In [ ]:
toy_velocity_probe = ch03.build_toy_velocity_probe(
    toy_model, toy_X0, toy_X1, device=DEVICE, seed=SEED + 4
)


In [ ]:
mean_conditional_velocity = np.asarray(toy_velocity_probe["mean_conditional_velocity"])
network_velocity = np.asarray(toy_velocity_probe["network_velocity"])
velocity_delta = network_velocity - mean_conditional_velocity
print(
    f"Probe neighborhood: n_local={len(toy_velocity_probe['local'])}, "
    f"t={toy_velocity_probe['t_value']:.2f}, "
    f"center=({toy_velocity_probe['center'][0]:.3f}, {toy_velocity_probe['center'][1]:.3f})"
)
print(
    "Mean conditional velocity vs network velocity: "
    f"conditional=({mean_conditional_velocity[0]:.3f}, {mean_conditional_velocity[1]:.3f}), "
    f"network=({network_velocity[0]:.3f}, {network_velocity[1]:.3f}), "
    f"delta=({velocity_delta[0]:.3f}, {velocity_delta[1]:.3f})"
)


In [ ]:
fig = ch03.plot_toy_conditional_vs_marginal(toy_model, toy_X0, toy_X1, toy_velocity_probe)
toy_fig32_path = save_and_show(fig, "fig03_02_conditional_vs_marginal_toy.png", width=620)


## 7. CFM Object Hierarchy

Figure 3.3 separates the objects that are easy to conflate when reading the loss. Panel A shows one endpoint pair and its conditional path; Panel B shows a population of independently sampled endpoint paths; Panel C shows the learned marginal field `v_theta(x,t)` over a grid at `t=0.5`. The plotting code is a display helper, while the conceptual hierarchy is listed here in the narrative.

In [ ]:
toy_hierarchy = ch03.prepare_toy_hierarchy_objects(
    toy_model, toy_X0, toy_X1, device=DEVICE, seed=SEED + 5
)
fig = ch03.plot_toy_cfm_object_hierarchy(toy_X0, toy_X1, toy_hierarchy)
toy_hierarchy_path = save_and_show(fig, "fig03_03_cfm_object_hierarchy_toy.png", width=900)


## Take-aways

- *Finding 1* — The toy CFM setup makes a multimodal source merge into a rotated central target using unordered endpoint samples rather than lineage labels.
- *Finding 2* — Linear conditional bridges make the velocity target easy to inspect: each sampled pair defines a local regression target for the neural field.
- *Finding 3* — Training uses local MSE without an ODE solve, while Euler integration appears only at inference to sample the learned field.

Next: → ch3_2
